# MILK10k — Step 3: Tri-modal fusion under missing modalities (Tier 1)
Runs on the **saved embeddings** from notebook 02 — no images, no backbone. A full config × 5 folds takes ~1–3 min on a T4 (CPU works too).

**Experiments**
* **E1 dedicated** — one model per availability pattern, trained only on that pattern (7 models): the reference ceiling.
* **E2 unified** — one tri-modal model evaluated under all 7 test patterns: {late, cross-attention} × {no dropout, modality dropout}.

**Leakage / shift guard.** Fold *f*'s embeddings of *training* lesions come from a backbone that trained on them (in-sample, extra-separable), while test lesions are out-of-sample. The ~10% of lesions the backbone used only for checkpoint selection are *out-of-sample for both C and D*, so they are used here as the fusion validation set (early stopping). Sanity check at the end: the unified model's C-only / D-only rows should be close to the notebook-02 unimodal CNNs.

In [ ]:
import os, glob, json, time, itertools, numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
import torch, torch.nn as nn, torch.nn.functional as F

# ---- where notebook 02 wrote its outputs (same session, or attached as input) ----
cands = glob.glob("/kaggle/working/stage2/emb_D_fold0.npy") + glob.glob("/kaggle/input/**/stage2/emb_D_fold0.npy", recursive=True)
assert cands, "attach the notebook-02 output (folder with stage2/) as an input"
STAGE2 = os.path.dirname(cands[0])
FOLDS_CSV = (glob.glob("/kaggle/working/milk10k_lesions_folds.csv") +
             glob.glob("/kaggle/input/**/milk10k_lesions_folds.csv", recursive=True))[0]
OUT = "/kaggle/working/stage3"; os.makedirs(OUT, exist_ok=True)
print("stage2:", STAGE2, "| folds:", FOLDS_CSV)

SEED = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); print(device)
les = pd.read_csv(FOLDS_CSV)
CLASSES = ['AKIEC','BCC','BEN_OTH','BKL','DF','INF','MAL_OTH','MEL','NV','SCCKA','VASC']; K = len(CLASSES)
y_all = les["y"].values.astype(int)
MODS = ["C", "D", "M"]
PATTERNS = ["CDM", "CD", "CM", "DM", "C", "D", "M"]                 # all 7 non-empty subsets
PATTERN_NAME = {"CDM": "full (clinic)", "CM": "no dermoscope (GP)", "C": "phone photo only",
                "D": "dermoscopy only", "CD": "images, no metadata", "DM": "no clinical photo", "M": "metadata only"}

## Inputs
Image tokens = backbone embedding **+ that image's own MONET scores** (so MONET vanishes with its image). Metadata token = age, skin tone, sex, site (+ missing flags). Fold-specific standardisation uses training lesions only.

In [ ]:
MONET = [c[:-2] for c in les.columns if c.startswith("MONET_") and c.endswith("_C")]
def load_emb(mod, f):
    return np.load(f"{STAGE2}/emb_{mod}_fold{f}.npy").astype(np.float32)

# metadata matrix (fold-independent encoding; scaling is done per fold below)
site = les["site"].fillna("unknown")
meta_raw = np.column_stack([
    les["age_approx"].fillna(-1).values, les["age_approx"].isna().values,
    les["skin_tone_class"].values,
    (les["sex"] == "male").values,
    pd.get_dummies(site).values]).astype(np.float32)
print("MONET per image:", len(MONET), "| metadata dims:", meta_raw.shape[1])

def fold_inputs(f):
    tr_mask = les["fold"].values != f
    X = {}
    for m in ["C", "D"]:
        X[m] = np.concatenate([load_emb(m, f), les[[f"{c}_{m}" for c in MONET]].values.astype(np.float32)], 1)
    X["M"] = meta_raw.copy()
    age = X["M"][:, 0]; age[age < 0] = np.median(age[tr_mask & (age >= 0)])
    for m in MODS:                                              # standardise with TRAIN lesions only
        mu, sd = X[m][tr_mask].mean(0), X[m][tr_mask].std(0) + 1e-6
        X[m] = (X[m] - mu) / sd
    return X

def backbone_val_ids(f):
    """Exactly the inner split notebook 02 used for checkpoint selection (out-of-sample embeddings)."""
    trval = les[les["fold"] != f]
    _, va = train_test_split(trval, test_size=0.10, stratify=trval["y"], random_state=SEED)
    return les.index.isin(va.index)

## Model
`late`: one head per modality, logits averaged over *available* modalities.  
`xattn`: modality tokens + [CLS] through a small Transformer; missing modalities are removed with the attention padding mask (not zero-filled).

In [ ]:
class FusionNet(nn.Module):
    def __init__(self, dims, kind="xattn", d=256, drop=0.3):
        super().__init__()
        self.kind = kind
        self.proj = nn.ModuleDict({m: nn.Sequential(nn.Dropout(0.2 if m != "M" else 0.0), nn.Linear(dims[m], d),
                                                    nn.GELU(), nn.LayerNorm(d)) for m in MODS})
        if kind == "late":
            self.heads = nn.ModuleDict({m: nn.Sequential(nn.Dropout(drop), nn.Linear(d, K)) for m in MODS})
        else:
            self.mod_emb = nn.Parameter(torch.randn(3, d) * 0.02)
            self.cls = nn.Parameter(torch.zeros(1, 1, d))
            layer = nn.TransformerEncoderLayer(d, nhead=4, dim_feedforward=2 * d, dropout=0.1,
                                               batch_first=True, norm_first=True)
            self.enc = nn.TransformerEncoder(layer, num_layers=2)
            self.head = nn.Sequential(nn.LayerNorm(d), nn.Dropout(drop), nn.Linear(d, K))

    def forward(self, x, avail):                      # avail: (B,3) bool in MODS order
        tok = torch.stack([self.proj[m](x[m]) for m in MODS], 1)          # (B,3,d)
        if self.kind == "late":
            logits = torch.stack([self.heads[m](tok[:, i]) for i, m in enumerate(MODS)], 1)
            w = avail.float().unsqueeze(-1)
            return (logits * w).sum(1) / w.sum(1).clamp(min=1)
        tok = tok + self.mod_emb
        seq = torch.cat([self.cls.expand(len(tok), -1, -1), tok], 1)
        pad = torch.cat([torch.zeros(len(tok), 1, dtype=torch.bool, device=tok.device), ~avail], 1)
        return self.head(self.enc(seq, src_key_padding_mask=pad)[:, 0])

def pattern_mask(p, n):
    return torch.tensor([[m in p for m in MODS]] * n, dtype=torch.bool, device=device)

NONEMPTY = torch.tensor([[m in p for m in MODS] for p in PATTERNS], dtype=torch.bool)

def sample_avail(n, mode, fixed=None):
    """mode: 'fixed' (dedicated), 'none' (always full), 'dropout' (50% full, else a uniform non-empty subset)."""
    if mode == "fixed": return pattern_mask(fixed, n)
    full = torch.ones(n, 3, dtype=torch.bool)
    if mode == "none": return full.to(device)
    sub = NONEMPTY[torch.randint(len(PATTERNS), (n,))]
    keep_full = torch.rand(n) < 0.5
    return torch.where(keep_full.unsqueeze(1), full, sub).to(device)

## Metrics + training loop

In [ ]:
def evaluate(y, P):
    pred = P.argmax(1)
    out = {"bal_acc": balanced_accuracy_score(y, pred),
           "macro_f1": f1_score(y, pred, average="macro", labels=range(K), zero_division=0),
           "macro_f1_thr05": np.mean([f1_score(y == k, P[:, k] >= 0.5, zero_division=0) for k in range(K)])}
    try: out["macro_auc"] = roc_auc_score(y, P, multi_class="ovr", average="macro", labels=range(K))
    except ValueError: out["macro_auc"] = np.nan
    return out

def to_t(X, idx): return {m: torch.tensor(X[m][idx], device=device) for m in MODS}

@torch.no_grad()
def predict(model, X, idx, pattern):
    model.eval(); x = to_t(X, idx)
    return model(x, pattern_mask(pattern, len(idx))).float().softmax(1).cpu().numpy()

def train_fusion(X, f, kind, mode, fixed=None, epochs=80, patience=15, bs=256, lr=3e-4):
    torch.manual_seed(SEED + f); np.random.seed(SEED + f)
    is_val = backbone_val_ids(f)
    tr_idx = np.where((les["fold"].values != f) & ~is_val)[0]
    va_idx = np.where(is_val)[0]
    val_pattern = fixed if mode == "fixed" else "CDM"
    n = np.bincount(y_all[tr_idx], minlength=K).astype(float)
    w = np.sqrt(n.max() / np.maximum(n, 1)); w = torch.tensor(w / w.mean(), dtype=torch.float, device=device)
    model = FusionNet({m: X[m].shape[1] for m in MODS}, kind).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    Xtr, ytr = to_t(X, tr_idx), torch.tensor(y_all[tr_idx], device=device)
    best, bad, state = -1, 0, None
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(tr_idx), device=device)
        for s in range(0, len(perm), bs):
            b = perm[s:s + bs]
            xb = {m: Xtr[m][b] for m in MODS}
            loss = F.cross_entropy(model(xb, sample_avail(len(b), mode, fixed)), ytr[b],
                                   weight=w, label_smoothing=0.05)
            opt.zero_grad(); loss.backward(); opt.step()
        # early stopping on out-of-sample lesions; with dropout, score = mean over all 7 patterns
        pats = [val_pattern] if mode != "dropout" else PATTERNS
        score = np.mean([evaluate(y_all[va_idx], predict(model, X, va_idx, p))["macro_f1"] for p in pats])
        if score > best: best, bad, state = score, 0, {k: v.clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(state)
    return model, ep + 1

## Run E1 (dedicated) + E2 (unified)

In [ ]:
CONFIGS = {f"dedicated_{p}": dict(kind="late", mode="fixed", fixed=p) for p in PATTERNS}
CONFIGS.update({
    "late_nodrop":  dict(kind="late",  mode="none"),
    "late_drop":    dict(kind="late",  mode="dropout"),
    "xattn_nodrop": dict(kind="xattn", mode="none"),
    "xattn_drop":   dict(kind="xattn", mode="dropout"),
})

rows, probs = [], {}          # probs[(config, pattern)] -> (5240, K) out-of-fold probabilities
for f in range(5):
    X = fold_inputs(f); te_idx = np.where(les["fold"].values == f)[0]; t0 = time.time()
    for name, cfg in CONFIGS.items():
        model, n_ep = train_fusion(X, f, **cfg)
        eval_patterns = [cfg["fixed"]] if cfg["mode"] == "fixed" else PATTERNS
        for p in eval_patterns:
            P = predict(model, X, te_idx, p)
            probs.setdefault((name, p), np.zeros((len(les), K), np.float32))[te_idx] = P
            rows.append(dict(config=name, fold=f, pattern=p, epochs=n_ep, **evaluate(y_all[te_idx], P)))
    print(f"fold {f} done in {time.time() - t0:.0f}s")

res = pd.DataFrame(rows); res.to_csv(f"{OUT}/tier1_fusion_results_long.csv", index=False)
np.savez_compressed(f"{OUT}/oof_probs_fusion.npz", **{f"{c}|{p}": v for (c, p), v in probs.items()})
print("saved", len(res), "rows")

## Main table — macro F1 per availability pattern (mean ± sd over 5 folds)

In [ ]:
def pm(metric):
    g = res.groupby(["config", "pattern"])[metric]
    t = (g.mean().round(3).astype(str) + " ± " + g.std().round(3).astype(str)).unstack("pattern")
    t = t.reindex(columns=PATTERNS)
    ded = pd.Series({p: t.loc[f"dedicated_{p}", p] for p in PATTERNS}, name="dedicated (per-pattern)")
    return pd.concat([ded.to_frame().T, t.drop(index=[f"dedicated_{p}" for p in PATTERNS])])
tab_f1 = pm("macro_f1"); tab_f1.to_csv(f"{OUT}/table_macro_f1.csv")
tab_f1.rename(columns=lambda p: f"{p}\n{PATTERN_NAME[p]}")

In [ ]:
tab_auc = pm("macro_auc"); tab_auc.to_csv(f"{OUT}/table_macro_auc.csv"); tab_auc


In [ ]:
tab_ba = pm("bal_acc"); tab_ba.to_csv(f"{OUT}/table_bal_acc.csv"); tab_ba


## Gap to dedicated + sanity check against notebook-02 CNNs

In [ ]:
mean = res.groupby(["config", "pattern"])["macro_f1"].mean().unstack().reindex(columns=PATTERNS)
ded = pd.Series({p: mean.loc[f"dedicated_{p}", p] for p in PATTERNS})
gap = (mean.drop(index=[f"dedicated_{p}" for p in PATTERNS]) - ded).round(3)
print("Unified minus dedicated (macro F1; 0 = no cost for using one model everywhere):")
display(gap)

print("\nSanity: notebook-02 unimodal CNNs (should be close to the C and D columns above)")
for m in ["C", "D"]:
    oof = pd.read_csv(f"{STAGE2}/oof_{m}.csv")
    print(m, {k: round(v, 3) for k, v in evaluate(oof["y"].values, oof[CLASSES].values).items()})